# Risk Models — Complete Ground-Up Guide
### For quant developer interviews | CS background assumed, finance background built from scratch

---

**What this notebook covers:**

1. What is risk? The vocabulary and intuitions every quant needs
2. Volatility: estimation, annualisation, EWMA dynamic models
3. The covariance matrix — why it is the central object of risk management
4. Sample covariance failure modes — the curse of dimensionality, Marchenko-Pastur
5. Ledoit-Wolf shrinkage — the analytical optimal fix
6. Factor risk models — PCA, macro factors, BARRA-style
7. Value at Risk (VaR) — parametric, historical, Monte Carlo
8. CVaR / Expected Shortfall — coherent risk, convex optimisation link
9. Stress testing and scenario analysis
10. Risk attribution — decomposing portfolio risk to sources

**Dependencies:**
```
pip install numpy scipy matplotlib scikit-learn cvxpy
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.linalg import eigh
from sklearn.covariance import LedoitWolf, OAS
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.color': 'white', 'grid.linewidth': 0.8,
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
})

PURPLE = '#534AB7'; TEAL = '#1D9E75'; AMBER = '#EF9F27'
CORAL  = '#D85A30'; BLUE = '#185FA5'; GREEN = '#3B6D11'
RED    = '#A32D2D'; GRAY = '#888780'; PINK  = '#D4537E'

np.random.seed(42)
print('Imports OK.')

---
## Part 1 — What Is Risk? Building the Right Mental Model

Coming from CS, you think of risk as probability of failure. In finance, risk is more nuanced: it is **the dispersion of future outcomes**. You don't know what an asset will return next year — risk models quantify the *shape* of that uncertainty.

### Two fundamentally different types of risk

**Systematic risk (market risk, undiversifiable):** Risk driven by broad market factors — economic growth, interest rates, inflation, credit conditions. If the global economy crashes, almost every risky asset falls together. No amount of diversification within risky assets eliminates this.

**Idiosyncratic risk (specific risk, diversifiable):** Risk unique to a specific company or instrument — a CEO scandal, a product recall, a patent dispute. This risk is uncorrelated across assets. Hold 50 assets and idiosyncratic risk is negligible.

**The core decomposition:**
$$r_i = \underbrace{\sum_k \beta_{ik} F_k}_{\text{systematic}} + \underbrace{\varepsilon_i}_{\text{idiosyncratic}}$$

This is the factor model — the backbone of all modern risk models.

### Why risk models matter for a robo-advisor (Scalable Capital context)

Scalable manages billions in ETF portfolios. Every client's risk profile (1–10) maps to a specific portfolio volatility target. The risk model tells the system:
- What is the *current* portfolio volatility given these weights?
- What happens to the portfolio in a 2008-style crash? (stress test)
- How much of the risk comes from equities vs bonds vs currency? (attribution)
- Are two ETFs so correlated that adding both provides no real diversification?

All of this flows from a well-estimated **covariance matrix** $\Sigma$.

In [ ]:
# ── Visualise systematic vs idiosyncratic risk decomposition ─────────────────
T = 252 * 3  # 3 years daily

# Simulate 2 factors: global equity, interest rates
F_equity = np.random.normal(0, 0.012, T)   # daily vol ~19% annual
F_rates  = np.random.normal(0, 0.004, T)   # daily vol ~6% annual

# Simulate 5 assets with known factor exposures
betas = np.array([
    [1.10,  0.10],   # Global equity ETF: high equity beta, small rates
    [0.85, -0.30],   # Balanced ETF
    [1.40,  0.05],   # EM equity ETF: high equity beta
    [0.05, -0.80],   # Government bond ETF: rate sensitive
    [0.30, -0.50],   # Corp bond ETF
])
asset_names = ['Global Equity', 'Balanced', 'EM Equity', 'Gov Bond', 'Corp Bond']
idio_vols   = np.array([0.004, 0.003, 0.007, 0.002, 0.003])  # daily idio vol

# Return = systematic + idiosyncratic
systematic  = np.column_stack([betas[:, 0]*F_equity + betas[:, 1]*F_rates
                                for _ in range(1)]).T  # shape (5, T)
systematic  = (F_equity * betas[:, 0:1] + F_rates * betas[:, 1:2]).T  # (T, 5)
idiosyncratic = np.random.normal(0, idio_vols, (T, 5))
total_rets  = systematic + idiosyncratic

# Variance decomposition
sys_var  = systematic.var(axis=0) * 252
idio_var = idiosyncratic.var(axis=0) * 252
tot_var  = total_rets.var(axis=0) * 252

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Stacked bar: variance decomposition per asset
x = np.arange(5)
axes[0].bar(x, sys_var * 100,  color=PURPLE, label='Systematic variance (%²)', alpha=0.85)
axes[0].bar(x, idio_var * 100, bottom=sys_var * 100, color=CORAL, label='Idiosyncratic variance (%²)', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(asset_names, rotation=20, fontsize=9)
axes[0].set_ylabel('Annual variance (%²)')
axes[0].set_title('Variance decomposition\nSystematic vs idiosyncratic')
axes[0].legend(fontsize=9)
for i in range(5):
    sys_pct = sys_var[i] / tot_var[i] * 100
    axes[0].text(i, tot_var[i]*100 + 0.01, f'{sys_pct:.0f}%\nsys', ha='center', fontsize=8)

# Factor return paths
axes[1].plot(np.cumsum(F_equity) * 100, color=PURPLE, lw=1.5, label='Global equity factor')
axes[1].plot(np.cumsum(F_rates)  * 100, color=BLUE,   lw=1.5, label='Interest rate factor')
axes[1].set_xlabel('Trading day')
axes[1].set_ylabel('Cumulative factor return (%)')
axes[1].set_title('Simulated factor return paths\n(drivers of systematic risk)')
axes[1].legend(fontsize=9)

# Portfolio diversification: add more assets
n_assets_range = np.arange(1, 51)
port_idio_var = []
for n in n_assets_range:
    avg_idio = np.mean(idio_vols[:min(n, 5)]**2) * 252
    port_idio = avg_idio / n   # idio variance falls as 1/n
    port_idio_var.append(port_idio)

sys_floor = np.mean(sys_var) * 0.85  # systematic risk floor
axes[2].plot(n_assets_range, np.array(port_idio_var) * 100 + sys_floor * 100,
             color=CORAL, lw=2.5, label='Total portfolio variance')
axes[2].axhline(sys_floor * 100, color=PURPLE, lw=1.5, ls='--',
                label=f'Systematic floor ({sys_floor*100:.1f}%²)')
axes[2].fill_between(n_assets_range,
                     np.array(port_idio_var)*100 + sys_floor*100,
                     sys_floor*100, alpha=0.2, color=CORAL, label='Diversifiable idio risk')
axes[2].set_xlabel('Number of assets in portfolio')
axes[2].set_ylabel('Annual portfolio variance (%²)')
axes[2].set_title('Diversification: idio risk vanishes\nsystematic risk cannot be diversified away')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Variance decomposition (% of total that is systematic):')
for name, sv, tv in zip(asset_names, sys_var, tot_var):
    print(f'  {name:15s}: {sv/tv*100:.0f}% systematic, {(1-sv/tv)*100:.0f}% idiosyncratic')

---
## Part 2 — Volatility: Static and Dynamic Estimation

Volatility $\sigma$ is the standard deviation of returns. It is the most fundamental single-number summary of risk.

### Static (rolling window) estimation
$$\hat{\sigma}_t = \sqrt{\frac{1}{T-1}\sum_{s=t-T+1}^{t}(r_s - \bar{r})^2}$$

**Problem:** all observations within the window get equal weight. A crash 250 days ago counts exactly as much as yesterday's crash. This produces a "ghost" effect — volatility spikes when a crash enters the window and again when it exits, even if markets have been calm in between.

### EWMA (Exponentially Weighted Moving Average)
Give more weight to recent observations:
$$\sigma^2_t = \lambda\,\sigma^2_{t-1} + (1-\lambda)\,r_{t-1}^2$$

$\lambda$ is the **decay factor** (typically 0.94 for daily data — RiskMetrics standard). Small $\lambda$ → fast-decaying memory (recent shocks dominate). Large $\lambda$ → long memory.

**Effective window:** $1/(1-\lambda)$ days. At $\lambda=0.94$: effective window ≈ 17 days.

### GARCH(1,1) — the gold standard for dynamic volatility
$$\sigma^2_t = \omega + \alpha\,r^2_{t-1} + \beta\,\sigma^2_{t-1}$$

where $\omega + \alpha + \beta < 1$ for stationarity. The long-run variance is $\sigma^2_{LR} = \omega/(1-\alpha-\beta)$.

GARCH captures **volatility clustering** — the empirical fact that large moves tend to cluster together (calm periods are calm, turbulent periods are turbulent). EWMA is a special case of GARCH with $\omega=0$, $\alpha=1-\lambda$, $\beta=\lambda$.

In [ ]:
# ── Simulate GARCH(1,1) process and compare vol estimators ───────────────────
def simulate_garch(T, omega, alpha, beta, mu=0):
    """Simulate GARCH(1,1) returns with time-varying volatility."""
    sigma2 = np.zeros(T)
    r      = np.zeros(T)
    sigma2[0] = omega / (1 - alpha - beta)   # start at long-run variance
    for t in range(1, T):
        sigma2[t] = omega + alpha * r[t-1]**2 + beta * sigma2[t-1]
        r[t]      = np.sqrt(sigma2[t]) * np.random.standard_normal()
    return r, np.sqrt(sigma2)

def ewma_vol(returns, lam=0.94):
    """EWMA volatility estimate."""
    T = len(returns)
    sigma2 = np.zeros(T)
    sigma2[0] = returns[0]**2
    for t in range(1, T):
        sigma2[t] = lam * sigma2[t-1] + (1 - lam) * returns[t-1]**2
    return np.sqrt(sigma2)

def rolling_vol(returns, window=30):
    """Rolling window volatility."""
    vol = np.full(len(returns), np.nan)
    for t in range(window, len(returns)):
        vol[t] = returns[t-window:t].std()
    return vol

# Simulate 5 years with a volatility spike in the middle
T = 252 * 5
omega, alpha, beta = 0.000002, 0.10, 0.88  # realistic GARCH params
np.random.seed(7)
rets, true_vol = simulate_garch(T, omega, alpha, beta)

# Inject a volatility regime change at year 2.5
mid = T // 2
rets[mid:mid+126], true_vol_spike = simulate_garch(126, omega*8, alpha, beta)
true_vol[mid:mid+126] = true_vol_spike

ewma_94  = ewma_vol(rets, lam=0.94)
ewma_97  = ewma_vol(rets, lam=0.97)
roll_30  = rolling_vol(rets, window=30)
roll_60  = rolling_vol(rets, window=60)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
days = np.arange(T)
ann  = np.sqrt(252)

# Main vol comparison
ax = axes[0, 0]
ax.plot(true_vol * ann * 100, color=GRAY,   lw=1, alpha=0.8, label='True GARCH σ')
ax.plot(ewma_94  * ann * 100, color=PURPLE, lw=1.5, label='EWMA λ=0.94 (fast)')
ax.plot(ewma_97  * ann * 100, color=TEAL,   lw=1.5, label='EWMA λ=0.97 (slow)')
ax.axvspan(mid, mid+126, alpha=0.1, color=RED, label='Volatility regime')
ax.set_title('EWMA volatility: fast vs slow decay')
ax.set_ylabel('Annualised vol (%)'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.plot(true_vol * ann * 100, color=GRAY,  lw=1, alpha=0.8, label='True GARCH σ')
ax.plot(roll_30  * ann * 100, color=CORAL, lw=1.5, label='Rolling 30d window')
ax.plot(roll_60  * ann * 100, color=BLUE,  lw=1.5, label='Rolling 60d window')
ax.axvspan(mid, mid+126, alpha=0.1, color=RED)
ax.set_title('Rolling window: ghost effect on exit\n(vol spikes again when crash leaves window)')
ax.set_ylabel('Annualised vol (%)'); ax.legend(fontsize=8)

# EWMA weights — show decay
ax = axes[1, 0]
lags = np.arange(60)
for lam, col, name in [(0.90, CORAL, 'λ=0.90'), (0.94, PURPLE, 'λ=0.94'), (0.97, TEAL, 'λ=0.97')]:
    weights = (1 - lam) * lam**lags
    weights /= weights.sum()
    ax.plot(lags, weights * 100, color=col, lw=2, label=f'{name} (eff window ≈ {1/(1-lam):.0f}d)')
ax.set_xlabel('Days ago')
ax.set_ylabel('Weight on observation (%)')
ax.set_title('EWMA observation weights\nRecent observations dominate')
ax.legend(fontsize=9)

# Distribution of GARCH returns vs normal
ax = axes[1, 1]
x_range = np.linspace(-0.08, 0.08, 300)
empirical_std = rets.std()
ax.hist(rets, bins=100, density=True, color=PURPLE, alpha=0.5, label='GARCH returns (simulated)')
ax.plot(x_range, stats.norm.pdf(x_range, 0, empirical_std), color=BLUE, lw=2,
        label='Normal distribution (same σ)')
ax.plot(x_range, stats.t.pdf(x_range, df=5, scale=empirical_std * np.sqrt(3/5)),
        color=CORAL, lw=2, label='Student-t df=5 (fat tails)')
ax.set_xlabel('Daily return')
ax.set_ylabel('Density')
ax.set_title('GARCH returns have fat tails\nNormal distribution underestimates extremes')
ax.legend(fontsize=8); ax.set_xlim(-0.07, 0.07)

# Compute empirical kurtosis
kurt = stats.kurtosis(rets)  # excess kurtosis; normal = 0
ax.text(0.98, 0.96, f'Excess kurtosis = {kurt:.2f}\n(normal = 0)', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color=PURPLE,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Dynamic volatility estimation: GARCH, EWMA, rolling window', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'GARCH process properties:')
print(f'  Long-run annual vol: {np.sqrt(omega/(1-alpha-beta)*252)*100:.1f}%')
print(f'  α + β = {alpha+beta:.2f}  (persistence — close to 1 = slow mean reversion)')
print(f'  Effective mean-reversion half-life: {np.log(0.5)/np.log(alpha+beta):.0f} days')

---
## Part 3 — The Covariance Matrix: Central Object of Risk Management

For $N$ assets with return vector $\mathbf{r}$, the covariance matrix is:
$$\Sigma_{ij} = \text{Cov}(r_i, r_j) = E[(r_i - \mu_i)(r_j - \mu_j)]$$

**Key properties every quant must know:**

1. **Symmetric:** $\Sigma = \Sigma^\top$ (since $\text{Cov}(i,j) = \text{Cov}(j,i)$)
2. **Positive semi-definite (PSD):** $\mathbf{x}^\top\Sigma\mathbf{x} \geq 0$ for all $\mathbf{x}$. All eigenvalues $\geq 0$.
3. **Portfolio variance:** $\sigma^2_p = \mathbf{w}^\top\Sigma\mathbf{w}$ — this is the quadratic form that makes portfolio optimisation a QP.
4. **Cholesky decomposition:** $\Sigma = LL^\top$ — used to simulate correlated returns efficiently.
5. **Spectral decomposition:** $\Sigma = Q\Lambda Q^\top$ — eigenvalues $\Lambda$ are the variances of principal components, eigenvectors $Q$ are the directions of maximum variance.

### The critical connection to portfolio volatility
$$\sigma_p = \sqrt{\mathbf{w}^\top\Sigma\mathbf{w}} = \sqrt{\sum_i\sum_j w_i w_j \sigma_i \sigma_j \rho_{ij}}$$

Every risk model computation — VaR, CVaR, risk attribution, optimisation — flows through $\Sigma$. Estimation quality here is everything.

### The curse of dimensionality
An $N\times N$ covariance matrix has $N(N+1)/2$ unique parameters. With $N=50$ ETFs: 1,275 parameters. With $T=252$ daily observations: you are estimating 1,275 parameters from 252 observations — **severely underdetermined.** The sample estimator is mathematically valid but statistically unreliable.

In [ ]:
# ── Covariance matrix structure, eigenvalues, Marchenko-Pastur noise bound ───
np.random.seed(42)

N = 40    # 40 assets
T = 252   # 1 year daily
q = T / N  # ratio — critical parameter

# True Sigma: factor structure (3 factors + idiosyncratic)
K = 3
B  = np.random.randn(N, K) * 0.08
Sf = np.diag([0.0200, 0.0050, 0.0015])   # factor variances (daily)
D  = np.diag(np.random.uniform(0.0005, 0.0030, N))
Sigma_true = B @ Sf @ B.T + D

# Simulate returns from true Sigma
L = np.linalg.cholesky(Sigma_true)
returns = np.random.randn(T, N) @ L.T

# Sample covariance
Sigma_sample = np.cov(returns.T)

# Eigenvalue spectra
eigvals_true   = np.sort(np.linalg.eigvalsh(Sigma_true))[::-1]
eigvals_sample = np.sort(np.linalg.eigvalsh(Sigma_sample))[::-1]

# Marchenko-Pastur distribution: bounds on noise eigenvalues
# For random matrix with T obs, N assets, variance sigma²:
sigma2_mp = 1.0 / N   # each return normalised to unit variance conceptually
# Using daily variance of the data:
var_mean = np.diag(Sigma_sample).mean()
lambda_plus  = var_mean * (1 + 1/np.sqrt(q))**2   # upper MP bound
lambda_minus = var_mean * (1 - 1/np.sqrt(q))**2   # lower MP bound

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Eigenvalue comparison
ax = axes[0]
ranks = np.arange(1, N+1)
ax.semilogy(ranks, eigvals_true,   'o-', color=GREEN,  lw=2, ms=4, label='True Σ eigenvalues')
ax.semilogy(ranks, eigvals_sample, 's-', color=CORAL,  lw=1.5, ms=4, label='Sample Σ eigenvalues')
ax.axhline(lambda_plus,  color=RED,    lw=1.5, ls='--', label=f'MP upper bound ({lambda_plus*252*100:.1f}% annual²)')
ax.axhline(lambda_minus, color=AMBER,  lw=1.5, ls='--', label=f'MP lower bound')
ax.fill_between(ranks, lambda_minus, lambda_plus, alpha=0.1, color=RED, label='MP noise band')
ax.set_xlabel('Eigenvalue rank')
ax.set_ylabel('Eigenvalue (log scale)')
ax.set_title(f'Eigenvalue spectrum: N={N}, T={T}, q=T/N={q:.1f}\nEigenvalues in noise band = estimation noise')
ax.legend(fontsize=8)

# Heatmap: sample vs true correlation matrix
def cov_to_corr(S):
    d = np.sqrt(np.diag(S))
    return S / np.outer(d, d)

corr_true   = cov_to_corr(Sigma_true)
corr_sample = cov_to_corr(Sigma_sample)

# Show first 15 assets for readability
n_show = 15
im = axes[1].imshow(corr_sample[:n_show, :n_show], cmap='RdYlGn', vmin=-0.8, vmax=0.8)
plt.colorbar(im, ax=axes[1], shrink=0.8)
axes[1].set_title(f'Sample correlation matrix (first {n_show} assets)\nNoisy off-diagonal elements')
axes[1].set_xlabel('Asset index'); axes[1].set_ylabel('Asset index')

# Frobenius error as T/N varies — showing breakdown
def frob_rel(A, B):
    return np.linalg.norm(A - B, 'fro') / np.linalg.norm(B, 'fro')

ratios = np.array([0.5, 1.0, 1.5, 2, 3, 5, 8, 10, 15, 20])
errors = []
for ratio in ratios:
    T_v = max(int(ratio * N), N + 2)
    R_v = np.random.randn(T_v, N) @ L.T
    Sv  = np.cov(R_v.T)
    errors.append(frob_rel(Sv, Sigma_true))

axes[2].plot(ratios, errors, 'o-', color=CORAL, lw=2.5, ms=7)
axes[2].axvline(1, color=RED, lw=1.5, ls='--', label='T=N: matrix barely invertible')
axes[2].axvline(q, color=AMBER, lw=1.5, ls='--', label=f'Our case T/N={q:.1f}')
axes[2].axhline(0.05, color=GREEN, lw=1, ls=':', label='5% error threshold')
axes[2].set_xlabel('T/N ratio (observations per asset)')
axes[2].set_ylabel('Relative Frobenius error')
axes[2].set_title('Sample Σ error vs data richness\nT/N < 5 → severe estimation error')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Signal vs noise eigenvalues
n_signal = (eigvals_sample > lambda_plus).sum()
print(f'Eigenvalue analysis (N={N}, T={T}, q=T/N={q:.1f}):')
print(f'  Sample eigenvalues above MP noise bound: {n_signal} (= number of detectable risk factors)')
print(f'  True number of factors: {K}')
print(f'  Relative estimation error: {frob_rel(Sigma_sample, Sigma_true):.3f} ({frob_rel(Sigma_sample, Sigma_true)*100:.1f}%)')
print(f'\nCondition number of sample Σ: {eigvals_sample.max()/max(eigvals_sample.min(),1e-12):.0f}')
print(f'Condition number of true Σ:   {eigvals_true.max()/eigvals_true.min():.0f}')
print(f'(High condition number → Σ⁻¹ is unstable → optimizer exploits noise)')

---
## Part 4 — Ledoit-Wolf Shrinkage: The Analytical Fix

The sample covariance is **unbiased** (correct in expectation) but has **high variance** (noisy realisations). We trade off some bias for much lower variance — this is the classic bias-variance tradeoff from ML applied to matrix estimation.

### The shrinkage estimator
$$\hat{\Sigma} = (1-\alpha)\,\Sigma_{sample} + \alpha\,F$$

$F$ is the **shrinkage target** — a structured, well-conditioned matrix. Common targets:
- **Scaled identity:** $F = \bar{\sigma}^2 I$ — assumes all assets have the same variance, zero covariance
- **Constant correlation:** $F_{ij} = \bar{\rho}\sigma_i\sigma_j$ — all pairs share the mean correlation
- **Factor model:** $F = B\hat{F}B^\top + \hat{D}$ — most structured, most assumptions

### The optimal $\alpha$
Ledoit & Wolf (2004) derived the **analytically optimal** $\alpha$ that minimises the expected squared Frobenius error $E[\|\hat{\Sigma} - \Sigma_{true}\|_F^2]$:
$$\alpha^* = \frac{\sum_{i,j}\text{AsympVar}(\hat{\Sigma}_{ij})}{\|\Sigma_{sample} - F\|_F^2}$$

This can be computed from the data without knowing $\Sigma_{true}$ — making it a practical, data-driven regularisation. In Python: `sklearn.covariance.LedoitWolf`.

### Why shrinkage works — the geometric intuition
Think of the sample covariance matrix living in a high-dimensional space. The true matrix is at some unknown point. The sample estimate overshoots in all directions — the largest sample eigenvalues are too large (overcounting variance in the directions of maximum variance), and the smallest are too small. Shrinkage pulls extreme eigenvalues toward the centre, reducing the overshoot.

In [ ]:
# ── Ledoit-Wolf shrinkage: multiple estimators compared ──────────────────────

# Fit multiple estimators
lw  = LedoitWolf().fit(returns)
oas = OAS().fit(returns)

Sigma_lw  = lw.covariance_
Sigma_oas = oas.covariance_
alpha_lw  = lw.shrinkage_
alpha_oas = oas.shrinkage_

# Manual identity-target shrinkage for comparison
def identity_shrinkage(S, alpha):
    mu = np.trace(S) / len(S)   # scaled identity target
    return (1 - alpha) * S + alpha * mu * np.eye(len(S))

Sigma_id50 = identity_shrinkage(Sigma_sample, 0.50)  # 50% shrinkage toward identity

# Errors
estimators = {
    'Sample Σ': (Sigma_sample, CORAL),
    'LW shrinkage': (Sigma_lw, PURPLE),
    'OAS shrinkage': (Sigma_oas, TEAL),
    'Id. shrinkage α=0.5': (Sigma_id50, AMBER),
}
errors = {k: frob_rel(v, Sigma_true) for k, (v, _) in estimators.items()}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Frobenius error bar chart
names_e = list(errors.keys())
vals_e  = list(errors.values())
colors_e = [v[1] for v in estimators.values()]
bars = axes[0].bar(names_e, [v*100 for v in vals_e], color=colors_e, edgecolor='white', width=0.6)
for bar, val in zip(bars, vals_e):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val*100:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Relative Frobenius error (%)')
axes[0].set_title(f'Estimation error vs true Σ\n(N={N}, T={T})')
axes[0].tick_params(axis='x', rotation=20)

# Eigenvalue comparison: shrinkage pulls extreme eigenvalues inward
eig_lw  = np.sort(np.linalg.eigvalsh(Sigma_lw))[::-1]
eig_id  = np.sort(np.linalg.eigvalsh(Sigma_id50))[::-1]

ranks = np.arange(1, N+1)
axes[1].semilogy(ranks, eigvals_true,   'o-', color=GREEN,  ms=4, lw=2,   label='True Σ')
axes[1].semilogy(ranks, eigvals_sample, 's-', color=CORAL,  ms=3, lw=1.5, label='Sample Σ', alpha=0.8)
axes[1].semilogy(ranks, eig_lw,         '^-', color=PURPLE, ms=3, lw=1.5, label=f'LW (α={alpha_lw:.3f})')
axes[1].semilogy(ranks, eig_id,         'D-', color=AMBER,  ms=3, lw=1.5, label='Id. shrink α=0.5')
axes[1].set_xlabel('Eigenvalue rank')
axes[1].set_ylabel('Eigenvalue (log)')
axes[1].set_title('Shrinkage pulls eigenvalue spectrum\ntoward the true spectrum')
axes[1].legend(fontsize=8)

# Sweep shrinkage alpha — show optimal
alphas = np.linspace(0, 1, 100)
frob_errors_sweep = []
for a in alphas:
    S_a = identity_shrinkage(Sigma_sample, a)
    frob_errors_sweep.append(frob_rel(S_a, Sigma_true))

opt_alpha_idx = np.argmin(frob_errors_sweep)
opt_alpha_val = alphas[opt_alpha_idx]

axes[2].plot(alphas, np.array(frob_errors_sweep)*100, color=PURPLE, lw=2.5)
axes[2].axvline(opt_alpha_val, color=AMBER, lw=2, ls='--',
                label=f'Optimal α = {opt_alpha_val:.2f}')
axes[2].axvline(alpha_lw, color=TEAL, lw=2, ls=':',
                label=f'LW analytical α = {alpha_lw:.3f}')
axes[2].axvline(0, color=CORAL, lw=1.5, ls=':', label=f'Sample (α=0): {frob_errors_sweep[0]*100:.1f}%')
axes[2].set_xlabel('Shrinkage intensity α')
axes[2].set_ylabel('Relative Frobenius error (%)')
axes[2].set_title('Optimal shrinkage via grid search\nLW formula finds it analytically')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Shrinkage results:')
for name, err in errors.items():
    improvement = (errors['Sample Σ'] - err) / errors['Sample Σ'] * 100
    print(f'  {name:25s}: error = {err*100:.2f}%  (improvement over sample: {improvement:+.1f}%)')
print(f'\nLedoit-Wolf shrinkage intensity: α = {alpha_lw:.4f}')
print(f'Interpretation: final Σ = {(1-alpha_lw)*100:.1f}% sample + {alpha_lw*100:.1f}% identity target')

---
## Part 5 — Factor Risk Models: PCA and Macro Factors

Factor models decompose the covariance matrix into **structured components**:
$$\Sigma = B\,F_{cov}\,B^\top + D$$

where $B$ is the $N\times K$ factor loading matrix, $F_{cov}$ is the $K\times K$ factor covariance, and $D$ is diagonal idiosyncratic variance. This reduces $N(N+1)/2$ parameters to $NK + K(K+1)/2 + N$ — a massive reduction.

### Two approaches to finding factors

**1. Statistical factors (PCA):** Extract factors purely from the return data — no economic assumptions. PCA finds the directions of maximum variance. The first principal component is the direction in return-space that explains the most cross-sectional variation. For equity ETFs, PC1 is almost always a "global market" factor.

**2. Fundamental / macro factors:** Use observable economic variables as pre-specified factors: global equity index return, interest rate change, credit spread, FX, commodity index. Each ETF's exposure ($\beta$) to each factor is estimated by OLS regression:
$$r_i = \alpha_i + \sum_{k=1}^{K}\beta_{ik}F_{k,t} + \varepsilon_{it}$$

The residuals $\varepsilon_{it}$ are the idiosyncratic returns — their variance forms the diagonal $D$ matrix.

### Why factor models dominate in production
- **Parsimony:** 5 factors for 50 assets → 250 + 15 + 50 = 315 parameters vs 1,275 for full matrix
- **Economic interpretability:** you know *why* two ETFs are correlated (both load on equity factor)
- **Stress testing:** you can shock specific factors ("what if interest rates rise 2%?")
- **Stability:** factor covariances are estimated from longer time-series; loadings are more stable

In [ ]:
# ── PCA factor model and macro factor model ──────────────────────────────────
from sklearn.linear_model import LinearRegression

# ─── PCA factor model ────────────────────────────────────────────────────────
pca = PCA(n_components=N)
pca.fit(returns)

# Factor scores (principal component returns) — shape (T, N)
factor_scores = pca.transform(returns)
loadings      = pca.components_.T   # shape (N, N) — columns are factor loadings

# Keep K=5 factors
K_pca = 5
B_pca = loadings[:, :K_pca]             # (N, K) factor loadings
F_scores = factor_scores[:, :K_pca]     # (T, K) factor return matrix
F_cov_pca = np.cov(F_scores.T)          # (K, K)

# Idiosyncratic residuals
reconstructed = F_scores @ B_pca.T
residuals_pca = returns - reconstructed
D_pca = np.diag(residuals_pca.var(axis=0))

Sigma_pca = B_pca @ F_cov_pca @ B_pca.T + D_pca

# ─── Macro factor model ──────────────────────────────────────────────────────
# Simulate observable macro factors
np.random.seed(1)
T2 = T
macro_factors = np.column_stack([
    np.random.normal(0, 0.012, T2),   # Global equity
    np.random.normal(0, 0.004, T2),   # Interest rates
    np.random.normal(0, 0.006, T2),   # Credit spread
    np.random.normal(0, 0.003, T2),   # USD index
    np.random.normal(0, 0.008, T2),   # Commodity
])
factor_names_macro = ['Global Equity', 'Int. Rates', 'Credit Spread', 'USD Index', 'Commodity']
K_macro = 5

# OLS regression of each asset on macro factors
B_macro    = np.zeros((N, K_macro))
alphas_m   = np.zeros(N)
resid_var  = np.zeros(N)
r2_values  = np.zeros(N)

for i in range(N):
    reg = LinearRegression()
    reg.fit(macro_factors, returns[:, i])
    B_macro[i] = reg.coef_
    alphas_m[i] = reg.intercept_
    pred = reg.predict(macro_factors)
    resid = returns[:, i] - pred
    resid_var[i] = resid.var()
    ss_tot = ((returns[:, i] - returns[:, i].mean())**2).sum()
    ss_res = (resid**2).sum()
    r2_values[i] = 1 - ss_res/ss_tot

F_cov_macro  = np.cov(macro_factors.T)
D_macro      = np.diag(resid_var)
Sigma_macro  = B_macro @ F_cov_macro @ B_macro.T + D_macro

# ─── Visualise ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# PCA variance explained
ax = axes[0, 0]
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.bar(range(1, 21), pca.explained_variance_ratio_[:20]*100,
       color=PURPLE, alpha=0.7, label='Individual %')
ax2 = ax.twinx()
ax2.plot(range(1, 21), cumvar[:20], 'o-', color=AMBER, lw=2, ms=5, label='Cumulative %')
ax2.axhline(80, color=TEAL, lw=1, ls='--', label='80% threshold')
ax2.set_ylabel('Cumulative variance explained (%)', color=AMBER)
ax.set_xlabel('Principal Component'); ax.set_ylabel('Variance explained (%)')
ax.set_title(f'PCA scree plot — first {K_pca} PCs explain {cumvar[K_pca-1]:.0f}% of variance')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, fontsize=8)

# Factor loading heatmap (first 5 PCs, first 10 assets)
ax = axes[0, 1]
n_show_assets = min(10, N)
im = ax.imshow(B_pca[:n_show_assets, :].T, cmap='RdBu_r', vmin=-0.3, vmax=0.3, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_yticks(range(K_pca))
ax.set_yticklabels([f'PC{i+1}' for i in range(K_pca)])
ax.set_xticks(range(n_show_assets))
ax.set_xticklabels([f'A{i+1}' for i in range(n_show_assets)])
ax.set_title(f'PCA factor loadings (first {K_pca} PCs)\nPC1 = global market factor (loads on all assets)')
ax.set_xlabel('Asset'); ax.set_ylabel('Factor')

# Macro factor R² values
ax = axes[1, 0]
ax.hist(r2_values, bins=20, color=TEAL, alpha=0.8, edgecolor='white')
ax.axvline(r2_values.mean(), color=CORAL, lw=2, ls='--',
           label=f'Mean R² = {r2_values.mean():.2f}')
ax.set_xlabel('R² (fraction of variance explained by macro factors)')
ax.set_ylabel('Number of assets')
ax.set_title(f'Macro factor model fit across {N} assets\nR² = systematic variance fraction')
ax.legend()

# Error comparison: sample vs PCA vs macro factor
ax = axes[1, 1]
errors_compare = {
    'Sample Σ': frob_rel(Sigma_sample, Sigma_true),
    f'LW Σ': frob_rel(Sigma_lw, Sigma_true),
    f'PCA {K_pca}-factor': frob_rel(Sigma_pca, Sigma_true),
    f'Macro {K_macro}-factor': frob_rel(Sigma_macro, Sigma_true),
}
colors_c = [CORAL, PURPLE, BLUE, TEAL]
bars = ax.bar(errors_compare.keys(), [v*100 for v in errors_compare.values()],
              color=colors_c, edgecolor='white', width=0.6)
for bar, val in zip(bars, errors_compare.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{val*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Relative Frobenius error (%)')
ax.set_title('Factor model vs shrinkage: estimation quality\n(lower = closer to true Σ)')
ax.tick_params(axis='x', rotation=15)

plt.suptitle('Factor risk models: PCA vs macro factor approach', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Model parameter count comparison (N={N}):')
print(f'  Full sample Σ:     {N*(N+1)//2:>6} parameters')
print(f'  PCA {K_pca}-factor model: {K_pca*N + K_pca*(K_pca+1)//2 + N:>6} parameters')
print(f'  Macro {K_macro}-factor:   {K_macro*N + K_macro*(K_macro+1)//2 + N:>6} parameters')
print(f'  Reduction vs full: {1 - (K_pca*N + K_pca*(K_pca+1)//2 + N)/(N*(N+1)//2):.0%}')

---
## Part 6 — VaR and CVaR: Tail Risk Measures

> **The formal VaR/CVaR treatment and the Artzner coherence axioms are in Module 02** ([`02_risk_measures_mpt.ipynb`](quant_finance_02_risk_measures_mpt.ipynb), §13). This section keeps only what is specific to a production risk model: the estimation methods and CVaR's optimisability.

### Three ways to estimate VaR
1. **Parametric (normal):** $\text{VaR}_\alpha = \mu - z_\alpha \sigma$, with $z_{95\%}=1.645$
2. **Historical simulation:** sort historical P&Ls, take the $(1-\alpha)$-th percentile
3. **Monte Carlo:** simulate scenarios from a model, take the tail percentile

VaR is **not sub-additive** (incoherent): combining two books can raise total VaR, contradicting diversification — the motivation for CVaR below.

### CVaR / Expected Shortfall — the production choice
$$\text{CVaR}_\alpha = E[\text{Loss}\mid \text{Loss}>\text{VaR}_\alpha]$$

CVaR is the average loss in the worst $(1-\alpha)\%$ of scenarios and is **coherent** (four axioms in Module 02). The key production fact: CVaR is **convex in portfolio weights** and can be minimised directly as a linear program via the Rockafellar-Uryasev (2000) formulation.

In [ ]:
# ── VaR and CVaR: three estimation methods, subadditivity failure ─────────────
np.random.seed(42)
T_sim = 10000
alpha = 0.95

# Two assets with fat-tailed returns, low correlation
mu_a, sig_a = 0.0003, 0.015
mu_b, sig_b = 0.0002, 0.010
rho_ab = 0.30

# Correlated Student-t (fat tails)
df = 4
cov_ab = np.array([[sig_a**2, rho_ab*sig_a*sig_b],
                    [rho_ab*sig_a*sig_b, sig_b**2]])
L_ab   = np.linalg.cholesky(cov_ab)
z = stats.t.rvs(df, size=(T_sim, 2))
z_scaled = z * np.sqrt((df-2)/df)   # scale so variance = 1
rets_2d = z_scaled @ L_ab.T + np.array([mu_a, mu_b])
rets_a, rets_b = rets_2d[:, 0], rets_2d[:, 1]
rets_port = 0.5 * rets_a + 0.5 * rets_b   # 50/50 portfolio

def historical_var_cvar(rets, alpha=0.95):
    sorted_rets = np.sort(rets)
    idx = int((1-alpha) * len(rets))
    var  = -sorted_rets[idx]
    cvar = -sorted_rets[:idx].mean()
    return var, cvar

def parametric_var_cvar(rets, alpha=0.95):
    mu, sig = rets.mean(), rets.std()
    z_alpha = stats.norm.ppf(1-alpha)
    var  = -(mu + z_alpha * sig)
    cvar = -(mu - sig * stats.norm.pdf(stats.norm.ppf(1-alpha)) / (1-alpha))
    return var, cvar

def monte_carlo_var_cvar(mu, sig, n_sims=50000, alpha=0.95):
    sims = np.random.normal(mu, sig, n_sims)
    return historical_var_cvar(sims, alpha)

var_a_h,  cvar_a_h  = historical_var_cvar(rets_a)
var_b_h,  cvar_b_h  = historical_var_cvar(rets_b)
var_p_h,  cvar_p_h  = historical_var_cvar(rets_port)
var_a_p,  cvar_a_p  = parametric_var_cvar(rets_a)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Return distributions with VaR/CVaR
for ax, rets, name, col in [
    (axes[0, 0], rets_a,    'Asset A',        PURPLE),
    (axes[0, 1], rets_port, '50/50 Portfolio', TEAL),
]:
    var_h, cvar_h = historical_var_cvar(rets)
    var_pm, cvar_pm = parametric_var_cvar(rets)
    counts, bins, _ = ax.hist(rets*100, bins=100, density=True, color=col, alpha=0.5,
                              label='Return distribution')
    tail_mask = bins[:-1] < -var_h*100
    ax.bar(bins[:-1][tail_mask], counts[tail_mask], width=np.diff(bins)[tail_mask],
           color=RED, alpha=0.8, align='edge', label=f'Worst {(1-alpha)*100:.0f}%')
    ax.axvline(-var_h  * 100, color=AMBER, lw=2, ls='--', label=f'VaR₉₅ hist={var_h*100:.2f}%')
    ax.axvline(-cvar_h * 100, color=RED,   lw=2, ls='-',  label=f'CVaR₉₅ hist={cvar_h*100:.2f}%')
    ax.axvline(-var_pm * 100, color=GRAY,  lw=1.5, ls=':', label=f'VaR₉₅ normal={var_pm*100:.2f}%')
    ax.set_xlabel('Daily return (%)')
    ax.set_ylabel('Density')
    ax.set_title(f'{name} — VaR vs CVaR (Student-t df={df})')
    ax.legend(fontsize=8)

# VaR sub-additivity failure
ax = axes[1, 0]
labels_sub = ['Asset A alone', 'Asset B alone', 'Sum of VaRs', '50/50 Portfolio']
var_vals   = [var_a_h, var_b_h, var_a_h + var_b_h, var_p_h]
cvar_vals  = [cvar_a_h, cvar_b_h, cvar_a_h + cvar_b_h, cvar_p_h]
x_sub = np.arange(4)
ax.bar(x_sub - 0.2, [v*100 for v in var_vals],  0.38, color=AMBER, label='VaR₉₅', alpha=0.85)
ax.bar(x_sub + 0.2, [v*100 for v in cvar_vals], 0.38, color=RED,   label='CVaR₉₅', alpha=0.85)
ax.axhline(var_a_h*100 + var_b_h*100, color=AMBER, lw=1.5, ls='--', alpha=0.6)
ax.set_xticks(x_sub); ax.set_xticklabels(labels_sub, rotation=15, fontsize=9)
ax.set_ylabel('Risk measure (%)')
ax.set_title('VaR: sub-additivity can fail\n"Sum of VaRs" vs actual portfolio VaR')
ax.legend(fontsize=9)
# Annotate sub-additivity failure/pass
if var_p_h < var_a_h + var_b_h:
    ax.annotate(f'VaR sub-additive\nhere (not guaranteed!)', (3, var_p_h*100),
                xytext=(2.0, (var_a_h+var_b_h)*100*0.7),
                fontsize=8, color=AMBER, arrowprops=dict(arrowstyle='->', color=AMBER))

# CVaR estimation methods comparison
ax = axes[1, 1]
alpha_range = np.linspace(0.90, 0.99, 30)
var_hist_r  = [historical_var_cvar(rets_a, a)[0] * 100 for a in alpha_range]
cvar_hist_r = [historical_var_cvar(rets_a, a)[1] * 100 for a in alpha_range]
var_norm_r  = [parametric_var_cvar(rets_a, a)[0] * 100 for a in alpha_range]
cvar_norm_r = [parametric_var_cvar(rets_a, a)[1] * 100 for a in alpha_range]

ax.plot(alpha_range*100, var_hist_r,  color=AMBER,  lw=2, label='VaR historical')
ax.plot(alpha_range*100, cvar_hist_r, color=RED,    lw=2, label='CVaR historical')
ax.plot(alpha_range*100, var_norm_r,  color=AMBER,  lw=1.5, ls='--', label='VaR normal')
ax.plot(alpha_range*100, cvar_norm_r, color=RED,    lw=1.5, ls='--', label='CVaR normal')
ax.fill_between(alpha_range*100,
                [abs(h-n) for h,n in zip(var_hist_r, var_norm_r)],
                alpha=0.15, color=AMBER, label='Model error (fat tails)')
ax.set_xlabel('Confidence level α (%)')
ax.set_ylabel('Risk measure (% daily loss)')
ax.set_title('VaR vs CVaR across confidence levels\nNormal model underestimates tail risk')
ax.legend(fontsize=8)

plt.suptitle('VaR and CVaR: theory, estimation, and coherence', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Risk measure comparison (95% confidence, daily):')
print(f'  Asset A:       VaR = {var_a_h*100:.3f}%   CVaR = {cvar_a_h*100:.3f}%')
print(f'  Asset B:       VaR = {var_b_h*100:.3f}%   CVaR = {cvar_b_h*100:.3f}%')
print(f'  Sum of risks:  VaR = {(var_a_h+var_b_h)*100:.3f}%   CVaR = {(cvar_a_h+cvar_b_h)*100:.3f}%')
print(f'  Portfolio:     VaR = {var_p_h*100:.3f}%   CVaR = {cvar_p_h*100:.3f}%')
print(f'  VaR sub-additive: {var_p_h < var_a_h + var_b_h} | CVaR sub-additive: {cvar_p_h < cvar_a_h + cvar_b_h}')
print(f'  (CVaR is ALWAYS sub-additive by the coherence theorem — VaR is NOT guaranteed)')

---
## Part 7 — Stress Testing and Risk Attribution

### Stress testing
Historical VaR and CVaR assume the future looks like the past. Stress tests ask: *what would happen to this portfolio in a specific extreme scenario?*

**Factor stress test:** Given a portfolio with factor loadings $\mathbf{b} = B^\top\mathbf{w}$, and a stress scenario $\Delta\mathbf{F}$ (e.g., equities fall 30%, rates rise 200bps), the stressed portfolio P&L is:
$$\text{P&L}_{stress} = \mathbf{b}^\top \Delta\mathbf{F} \cdot V_{portfolio}$$

### Risk attribution
The marginal risk contribution of asset $i$ to total portfolio volatility:
$$\text{MRC}_i = \frac{(\Sigma\mathbf{w})_i}{\sigma_p}$$

The total risk contribution:
$$\text{RC}_i = w_i \cdot \text{MRC}_i \qquad \sum_i \text{RC}_i = \sigma_p$$

**Factor risk attribution:** decompose $\sigma^2_p = \mathbf{w}^\top\Sigma\mathbf{w}$ using $\Sigma = BF_{cov}B^\top + D$:
$$\sigma^2_p = \underbrace{\mathbf{b}^\top F_{cov}\mathbf{b}}_{\text{factor risk}} + \underbrace{\mathbf{w}^\top D\mathbf{w}}_{\text{idiosyncratic risk}}$$

where $\mathbf{b} = B^\top\mathbf{w}$ is the portfolio's exposure vector to each factor.

In [ ]:
# ── Stress testing and risk attribution ─────────────────────────────────────

# Use factor model from Part 5
# Macro factors: [Global Equity, Int. Rates, Credit Spread, USD Index, Commodity]

# Define a 5-asset portfolio (with meaningful names for illustration)
port_assets  = ['Global Eq ETF', 'EM Eq ETF', 'Gov Bond ETF', 'Corp Bond ETF', 'Gold ETF']
n_port       = 5
w_port       = np.array([0.35, 0.15, 0.20, 0.15, 0.15])   # portfolio weights

# Define factor loadings for these 5 assets manually (economically meaningful)
# Columns: [Global Equity, Int. Rates, Credit Spread, USD Index, Commodity]
B_port = np.array([
    [ 0.95, -0.05, -0.10, -0.05,  0.02],  # Global Equity ETF: high equity, slight neg rates
    [ 1.20, -0.02, -0.20, -0.15,  0.05],  # EM Equity: higher equity beta, FX sensitive
    [-0.05, -0.90,  0.05,  0.10,  0.00],  # Gov Bond: very rate sensitive
    [ 0.20, -0.50, -0.60,  0.05,  0.00],  # Corp Bond: credit and rate sensitive
    [ 0.10,  0.10,  0.05, -0.30,  0.80],  # Gold: commodity and safe-haven
])

# Portfolio factor exposure
b_port = B_port.T @ w_port   # (5,) portfolio exposures to each macro factor

# Simple covariance for 5-asset portfolio (annual)
factor_ann_vols = np.array([0.190, 0.063, 0.095, 0.048, 0.126])
factor_corr = np.array([
    [1.00, -0.30,  0.20,  0.10, -0.05],
    [-0.30, 1.00, -0.20,  0.15,  0.10],
    [ 0.20,-0.20,  1.00, -0.05, -0.10],
    [ 0.10, 0.15, -0.05,  1.00, -0.02],
    [-0.05, 0.10, -0.10, -0.02,  1.00],
])
F_cov_ann = np.outer(factor_ann_vols, factor_ann_vols) * factor_corr
idio_vols_port = np.array([0.040, 0.080, 0.015, 0.025, 0.060])   # annual
D_port_ann = np.diag(idio_vols_port**2)
Sigma_port = B_port @ F_cov_ann @ B_port.T + D_port_ann

port_vol = np.sqrt(w_port @ Sigma_port @ w_port)

# Stress scenarios: [Global Equity, Int. Rates, Credit Spread, USD Index, Commodity]
scenarios = {
    '2008 Financial Crisis':   np.array([-0.40,  0.003, 0.040,  0.08, -0.30]),
    'COVID Mar 2020':          np.array([-0.34, -0.007, 0.015,  0.05, -0.10]),
    'Rate Shock (+200bps)':    np.array([ 0.00,  0.020, 0.005, -0.02,  0.02]),
    'EM Crisis':               np.array([-0.20, -0.005, 0.030,  0.10,  0.05]),
    'Inflation Spike':         np.array([-0.10,  0.015, 0.010, -0.05,  0.20]),
    'Tech Crash (mild)':       np.array([-0.15, -0.003, 0.005,  0.02,  0.05]),
}

stressed_pnl = {name: float(b_port @ shocks) for name, shocks in scenarios.items()}

# Risk attribution
MRC = (Sigma_port @ w_port) / port_vol   # marginal risk contribution
RC  = w_port * MRC                        # total risk contribution

# Factor risk attribution
factor_risk_total = b_port @ F_cov_ann @ b_port   # systematic variance
idio_risk_total   = w_port @ D_port_ann @ w_port   # idiosyncratic variance
factor_per_factor = b_port**2 * np.diag(F_cov_ann)  # approx per-factor contribution

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Stress test results
ax = axes[0, 0]
scenario_names = list(stressed_pnl.keys())
pnl_vals = list(stressed_pnl.values())
colors_stress = [RED if v < 0 else GREEN for v in pnl_vals]
bars = ax.barh(scenario_names, [v*100 for v in pnl_vals],
               color=colors_stress, edgecolor='white', height=0.6)
ax.axvline(0, color='black', lw=1)
for bar, val in zip(bars, pnl_vals):
    ax.text(val*100 + (0.3 if val >= 0 else -0.3), bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
ax.set_xlabel('Stressed portfolio P&L (%)')
ax.set_title('Stress test results\n(factor model × scenario shocks)')

# Risk attribution by asset
ax = axes[0, 1]
rc_pct = RC / port_vol * 100
colors_rc = [PURPLE, CORAL, BLUE, TEAL, AMBER]
bars = ax.bar(port_assets, rc_pct, color=colors_rc, edgecolor='white', width=0.6)
ax.axhline(100/n_port, color=GRAY, lw=1.5, ls='--', label=f'Equal ({100/n_port:.0f}% each)')
for bar, val, w in zip(bars, rc_pct, w_port*100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.0f}%\nw={w:.0f}%', ha='center', fontsize=8)
ax.set_ylabel('Risk contribution (% of portfolio σ)')
ax.set_title(f'Asset risk attribution\n(portfolio σ = {port_vol*100:.1f}%/year)')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=15)

# Factor risk attribution
ax = axes[1, 0]
factor_contribs_pct = factor_per_factor / (port_vol**2) * 100
idio_contrib_pct    = idio_risk_total / (port_vol**2) * 100
labels_f = factor_names_macro + ['Idiosyncratic']
sizes_f  = list(factor_contribs_pct) + [idio_contrib_pct]
colors_f = [PURPLE, BLUE, CORAL, TEAL, AMBER, GRAY]
wedges, texts, autotexts = ax.pie(
    [max(s, 0) for s in sizes_f], labels=labels_f, colors=colors_f,
    autopct='%1.0f%%', startangle=90, textprops={'fontsize': 9}
)
ax.set_title(f'Portfolio variance attribution by risk source\n(total σ² = {port_vol**2*100:.2f}%²)')

# Tornado chart: factor sensitivity
ax = axes[1, 1]
factor_vols_ann = np.sqrt(np.diag(F_cov_ann))
sensitivities_1sigma = b_port * factor_vols_ann * 100   # 1-sigma shock impact on portfolio (%)
sorted_idx = np.argsort(np.abs(sensitivities_1sigma))[::-1]
bar_colors_s = [RED if s < 0 else GREEN for s in sensitivities_1sigma[sorted_idx]]
ax.barh(
    [factor_names_macro[i] for i in sorted_idx],
    sensitivities_1sigma[sorted_idx],
    color=bar_colors_s, edgecolor='white', height=0.6
)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Portfolio impact of 1-σ factor shock (%)')
ax.set_title('Factor sensitivity tornado chart\n(1-sigma shock to each factor)')

plt.suptitle('Stress testing and risk attribution', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Portfolio annual volatility: {port_vol*100:.2f}%')
print(f'Systematic variance share: {factor_risk_total/port_vol**2*100:.0f}%')
print(f'Idiosyncratic variance share: {idio_risk_total/port_vol**2*100:.0f}%')
print('\nFactor sensitivities (portfolio % per 1σ factor shock):')
for fname, sens in zip(factor_names_macro, sensitivities_1sigma):
    bar = '█' * int(abs(sens) * 3)
    print(f'  {fname:20s}: {sens:+.2f}%  {bar}')

---
## Summary: Risk Model Mental Map

| Layer | Concept | Key equation | Tool |
|---|---|---|---|
| Return dynamics | GARCH / EWMA | $\sigma^2_t = \omega + \alpha r^2_{t-1} + \beta\sigma^2_{t-1}$ | scipy, arch |
| Covariance | Full matrix | $\Sigma_{ij} = E[(r_i-\mu_i)(r_j-\mu_j)]$ | numpy |
| Regularisation | LW shrinkage | $\hat{\Sigma} = (1-\alpha)\Sigma_s + \alpha F$ | sklearn |
| Factor model | Decomposition | $\Sigma = BF_{cov}B^\top + D$ | sklearn PCA / OLS |
| Tail risk (incoherent) | VaR | $P(\text{Loss} > VaR_\alpha) = 1-\alpha$ | numpy.percentile |
| Tail risk (coherent) | CVaR | $E[\text{Loss}\mid\text{Loss}>VaR_\alpha]$ | cvxpy |
| Stress test | Factor shock | $\text{P\&L} = \mathbf{b}^\top\Delta F \cdot V$ | numpy |
| Attribution | Risk contrib | $RC_i = w_i(\Sigma\mathbf{w})_i/\sigma_p$ | numpy |

**The three things to say in any interview about risk models:**
1. The covariance matrix is the central object — everything flows through $\mathbf{w}^\top\Sigma\mathbf{w}$
2. The sample covariance is unreliable when $T/N < 10$ — use Ledoit-Wolf shrinkage or a factor model
3. CVaR is the correct tail risk measure — it is coherent (sub-additive) and convex in weights, enabling direct optimisation